# Explicit Fraud-Detection Analysis

This notebook analyzes the full 56,962-row classical held-out test set from saved predictions. It does not retrain any model or alter thresholds.

## Operational meanings

- True Positive: fraud correctly detected.
- False Negative: fraud missed.
- False Positive: legitimate transaction incorrectly flagged.
- True Negative: legitimate transaction correctly accepted.

The models use their existing native prediction rules. No threshold optimization is performed.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RESULTS_PATH = PROJECT_ROOT / 'results' / 'json' / 'classical_results.json'
PREDICTIONS_PATH = PROJECT_ROOT / 'results' / 'json' / 'classical_test_predictions.json'
DATASET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'
OUTPUT_PATH = PROJECT_ROOT / 'results' / 'json' / 'fraud_detection_analysis.json'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures' / 'fraud_detection'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
classical_results = json.loads(RESULTS_PATH.read_text())
prediction_rows = json.loads(PREDICTIONS_PATH.read_text())
predictions = pd.DataFrame(prediction_rows)
assert len(predictions) == 56962
print('Loaded rows:', len(predictions))

Loaded rows: 56962


## Saved prediction schema

The analysis uses the saved source row index, `Time`, true class, native predictions, and ranking scores. Amount is joined from the unchanged raw dataset only for representative display rows.

In [2]:
MODEL_KEYS = {
    'Logistic Regression': ('logistic_regression_prediction', 'logistic_regression_score'),
    'RBF-SVM': ('rbf_svm_prediction', 'rbf_svm_score'),
    'XGBoost': ('xgboost_prediction', 'xgboost_score'),
}
analysis_rows = []
detection_rows = []
for model, (prediction_column, score_column) in MODEL_KEYS.items():
    y_true = predictions['true_class'].to_numpy()
    y_pred = predictions[prediction_column].to_numpy()
    y_score = predictions[score_column].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    row = {
        'model': model, 'total_test_transactions': len(y_true),
        'actual_frauds': int((y_true == 1).sum()), 'actual_legitimate': int((y_true == 0).sum()),
        'predicted_frauds': int((y_pred == 1).sum()), 'predicted_legitimate': int((y_pred == 0).sum()),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'F1': float(f1_score(y_true, y_pred, zero_division=0)),
        'PR-AUC': float(average_precision_score(y_true, y_score)),
        'ROC-AUC': float(roc_auc_score(y_true, y_score)),
        'specificity': float(tn / (tn + fp)), 'FPR': float(fp / (fp + tn)),
    }
    analysis_rows.append(row)
    detection_rows.append({'model': model, 'frauds actually present': int(tp + fn), 'frauds detected': int(tp), 'frauds missed': int(fn), 'fraud detection recall': row['recall'], 'legitimate transactions falsely flagged': int(fp)})
analysis_table = pd.DataFrame(analysis_rows)
detection_table = pd.DataFrame(detection_rows)
display(analysis_table)
display(detection_table)

,model,total_test_transactions,actual_frauds,actual_legitimate,predicted_frauds,predicted_legitimate,TP,FP,FN,TN,precision,recall,F1,PR-AUC,ROC-AUC,specificity,FPR
0,Logistic Regression,56962,98,56864,1476,55486,90,1386,8,55478,0.060976,0.918367,0.114358,0.718971,0.972083,0.975626,0.024374
1,RBF-SVM,56962,98,56864,230,56732,74,156,24,56708,0.321739,0.755102,0.451220,0.477780,0.973191,0.997257,0.002743
2,XGBoost,56962,98,56864,95,56867,83,12,15,56852,0.873684,0.846939,0.860104,0.888891,0.978681,0.999789,0.000211


,model,frauds actually present,frauds detected,frauds missed,fraud detection recall,legitimate transactions falsely flagged
0,Logistic Regression,98,90,8,0.918367,1386
1,RBF-SVM,98,74,24,0.755102,156
2,XGBoost,98,83,15,0.846939,12


In [3]:
for row in analysis_rows:
    saved = classical_results['models'][row['model'].lower().replace('-', '_').replace(' ', '_')]
    assert np.isclose(row['PR-AUC'], saved['pr_auc'])
    assert np.isclose(row['ROC-AUC'], saved['roc_auc'])
    assert row['TP'] == saved['confusion_matrix']['true_positive']
    assert row['FP'] == saved['confusion_matrix']['false_positive']
    assert row['FN'] == saved['confusion_matrix']['false_negative']
    assert row['TN'] == saved['confusion_matrix']['true_negative']
print('Saved metric and confusion-matrix checks passed.')

Saved metric and confusion-matrix checks passed.


In [4]:
for row in analysis_rows:
    model_slug = row['model'].lower().replace('-', '_').replace(' ', '_')
    matrix = np.array([[row['TN'], row['FP']], [row['FN'], row['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Predicted legitimate', 'Predicted fraud'], yticklabels=['Actual legitimate', 'Actual fraud'], ax=ax)  # type: ignore[arg-type]
    ax.set_title(f'{row["model"]} Confusion Matrix')
    ax.set_xlabel('Prediction'); ax.set_ylabel('Actual class')
    fig.tight_layout(); fig.savefig(FIGURES_DIR / f'{model_slug}_confusion_matrix.png', dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(analysis_rows)); width = 0.25
ax.bar(x - width, [r['actual_frauds'] for r in analysis_rows], width, label='Frauds actually present')
ax.bar(x, [r['TP'] for r in analysis_rows], width, label='Frauds detected')
ax.bar(x + width, [r['FN'] for r in analysis_rows], width, label='Frauds missed')
ax.set_xticks(x, [r['model'] for r in analysis_rows]); ax.set_ylabel('Transactions'); ax.set_title('Actual Fraud vs Detected vs Missed'); ax.legend(); fig.tight_layout(); fig.savefig(FIGURES_DIR / 'fraud_detected_missed.png', dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([r['model'] for r in analysis_rows], [r['FP'] for r in analysis_rows], color='#d95f02')
ax.set_ylabel('Legitimate transactions falsely flagged'); ax.set_title('False Positives by Classical Model'); ax.tick_params(axis='x', rotation=15); fig.tight_layout(); fig.savefig(FIGURES_DIR / 'legitimate_transactions_flagged.png', dpi=200); plt.close(fig)

In [5]:
for row in analysis_rows:
    model_slug = row['model'].lower().replace('-', '_').replace(' ', '_')
    matrix = np.array([[row['TN'], row['FP']], [row['FN'], row['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Predicted legitimate', 'Predicted fraud'], yticklabels=['Actual legitimate', 'Actual fraud'], ax=ax)  # type: ignore[arg-type]
    ax.set_title(f"{row['model']} Confusion Matrix")
    ax.set_xlabel('Prediction'); ax.set_ylabel('Actual class')
    fig.tight_layout(); fig.savefig(FIGURES_DIR / f'{model_slug}_confusion_matrix.png', dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(analysis_rows)); width = 0.25
ax.bar(x - width, [r['actual_frauds'] for r in analysis_rows], width, label='Frauds actually present')
ax.bar(x, [r['TP'] for r in analysis_rows], width, label='Frauds detected')
ax.bar(x + width, [r['FN'] for r in analysis_rows], width, label='Frauds missed')
ax.set_xticks(x, [r['model'] for r in analysis_rows]); ax.set_ylabel('Transactions'); ax.set_title('Actual Fraud vs Detected vs Missed'); ax.legend(); fig.tight_layout(); fig.savefig(FIGURES_DIR / 'fraud_detected_missed.png', dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([r['model'] for r in analysis_rows], [r['FP'] for r in analysis_rows], color='#d95f02')
ax.set_ylabel('Legitimate transactions falsely flagged'); ax.set_title('False Positives by Classical Model'); ax.tick_params(axis='x', rotation=15); fig.tight_layout(); fig.savefig(FIGURES_DIR / 'legitimate_transactions_flagged.png', dpi=200); plt.close(fig)

In [6]:
raw = pd.read_csv(DATASET_PATH, usecols=['Time', 'Amount', 'Class'])
representative = predictions.sort_values('source_row_index').head(12).merge(raw, left_on='source_row_index', right_index=True, how='left')
representative = representative[['source_row_index', 'Time', 'Amount', 'Class', 'logistic_regression_prediction', 'logistic_regression_score', 'rbf_svm_prediction', 'rbf_svm_score', 'xgboost_prediction', 'xgboost_score']].rename(columns={'source_row_index': 'transaction index', 'Class': 'actual Class'})
representative.to_csv(PROJECT_ROOT / 'results' / 'tables' / 'representative_heldout_transactions.csv', index=False)
display(representative)

,transaction index,Time,Amount,actual Class,logistic_regression_prediction,logistic_regression_score,rbf_svm_prediction,rbf_svm_score,xgboost_prediction,xgboost_score
55747,5,2.0,3.67,0,0,0.033758,0,-1.719311,0,2.818894e-07
39537,9,9.0,3.68,0,0,0.019614,0,-2.537299,0,1.004161e-06
16652,10,10.0,7.80,0,0,0.016833,0,-2.900335,0,5.214436e-07
10531,13,11.0,27.50,0,0,0.328416,0,-1.413371,0,1.088374e-05
46440,15,12.0,15.99,0,0,0.005600,0,-3.885353,0,1.160268e-06
38336,18,14.0,46.80,0,1,0.605341,0,-1.788823,0,3.593497e-05
40951,19,15.0,5.00,0,0,0.013898,0,-3.025444,0,4.017612e-07
37833,36,27.0,30.49,0,0,0.143766,0,-3.329928,0,8.816993e-07
290,37,27.0,1.80,0,0,0.009344,0,-2.745511,0,6.164760e-08
50659,40,32.0,29.89,0,0,0.006536,0,-2.642808,0,6.604902e-07


In [7]:
artifact = {
    'analysis_scope': 'Full 56,962-row classical held-out baseline test set',
    'prediction_source': str(PREDICTIONS_PATH.relative_to(PROJECT_ROOT)),
    'threshold_policy': classical_results['threshold_policy'],
    'models': analysis_rows,
    'fraud_detection_table': detection_rows,
    'representative_transactions_file': 'results/tables/representative_heldout_transactions.csv',
    'figures': sorted(path.name for path in FIGURES_DIR.iterdir() if path.is_file()),
    'limitations': ['Uses existing native prediction rules; no threshold optimization.', 'Representative rows are held-out examples and do not prove general performance.', 'This analysis does not retrain models.'],
}
OUTPUT_PATH.write_text(json.dumps(artifact, indent=2) + '\n')
print('Saved', OUTPUT_PATH)

Saved /Users/sakshamvarshney/Developer/qml-fraud-detection/results/json/fraud_detection_analysis.json


## Interpretation

These counts describe operational tradeoffs on the existing held-out test set. They do not rank the models or establish a winner. PR-AUC remains the primary metric for the imbalanced problem, and representative transactions are illustrative held-out examples rather than evidence of general performance.

In [8]:
assert OUTPUT_PATH.exists()
assert len(predictions) == 56962
assert all((FIGURES_DIR / name).exists() for name in artifact['figures'])
print('Analysis complete: 56,962 predictions verified; no model retraining performed.')

Analysis complete: 56,962 predictions verified; no model retraining performed.
